# Extreme Climate Events Analysis: 02_Feature_Engineering


## Initial Setup and Data Loading

In [ ]:
# Using the query function to load data from the database.
import sys
sys.path.append('../src')
from database import query

ready


In [ ]:
# Load the data from the database using the query function. We select only the relevant columns for our analysis.
df = query("""
    SELECT EVENT_ID,
           STATE,
           YEAR,
           MONTH_NAME,
           EVENT_TYPE,
           DAMAGE_PROPERTY,
           DAMAGE_CROPS,
           DEATHS_DIRECT,
           DEATHS_INDIRECT
    FROM details
""")

print(df.shape)

(1520485, 9)


In [7]:
import pandas as pd
import numpy as np

def convert_damage(value):
    """Convert NOAA damage format (1.5K, 2M, 3B) to numeric."""
    if pd.isna(value):
        return 0
    value = str(value).upper().strip()
    if value in ('', '0', '0.00'):
        return 0
    try:
        if 'K' in value:
            return float(value.replace('K', '')) * 1_000
        elif 'M' in value:
            return float(value.replace('M', '')) * 1_000_000
        elif 'B' in value:
            return float(value.replace('B', '')) * 1_000_000_000
        else:
            return float(value)
    except (ValueError, AttributeError):
        return 0

df['DAMAGE_PROPERTY_NUM'] = df['DAMAGE_PROPERTY'].apply(convert_damage)
df['DAMAGE_CROPS_NUM']    = df['DAMAGE_CROPS'].apply(convert_damage)
df['TOTAL_DAMAGE']        = df['DAMAGE_PROPERTY_NUM'] + df['DAMAGE_CROPS_NUM']
df['TOTAL_DEATHS']        = df['DEATHS_DIRECT'] + df['DEATHS_INDIRECT']

print("Damage columns converted.")
print(df[['DAMAGE_PROPERTY', 'DAMAGE_PROPERTY_NUM', 'TOTAL_DAMAGE', 'TOTAL_DEATHS']].head())

Damage columns converted.
  DAMAGE_PROPERTY  DAMAGE_PROPERTY_NUM  TOTAL_DAMAGE  TOTAL_DEATHS
0             NaN                  0.0           0.0             0
1             NaN                  0.0           0.0             0
2             NaN                  0.0           0.0             0
3             NaN                  0.0           0.0             0
4              2K               2000.0        2000.0             0


## Feature 01: Annual Event Frequency per State


In [11]:
# Create an annual event frequency per state, which will be our target variable for modeling.
annual_freq = (
    df.groupby(['STATE', 'YEAR'])
    .size()
    .reset_index(name='ANNUAL_EVENT_COUNT')
)

print(annual_freq.head(10))
print(f"\nShape: {annual_freq.shape}")

     STATE  YEAR  ANNUAL_EVENT_COUNT
0  ALABAMA  2000                1008
1  ALABAMA  2001                 765
2  ALABAMA  2002                 759
3  ALABAMA  2003                1143
4  ALABAMA  2004                 848
5  ALABAMA  2005                1332
6  ALABAMA  2006                1143
7  ALABAMA  2007                1316
8  ALABAMA  2008                1695
9  ALABAMA  2009                1756

Shape: (1659, 3)


In [10]:
# Merge annual frequency back into main dataframe
df = df.merge(annual_freq, on=['STATE', 'YEAR'], how='left')

print(df[['STATE', 'YEAR', 'ANNUAL_EVENT_COUNT']].head())

           STATE  YEAR  ANNUAL_EVENT_COUNT
0        FLORIDA  2000              1244.0
1        FLORIDA  2000              1244.0
2        FLORIDA  2000              1244.0
3  WEST VIRGINIA  2000              1123.0
4    MISSISSIPPI  2000               934.0


## Feature 02: Moving average of events


In [15]:
# 3-year rolling average of events per state
rolling_avg = (
    annual_freq
    .sort_values(['STATE', 'YEAR'])
    .groupby('STATE')['ANNUAL_EVENT_COUNT']
    .rolling(window=3, min_periods=1)
    .mean()
    .reset_index()
    .rename(columns={'ANNUAL_EVENT_COUNT': 'ROLLING_AVG_3Y'})
)

# The groupby+rolling creates a multi-index, fix it
rolling_avg = rolling_avg.drop(columns='level_1', errors='ignore')
rolling_avg['STATE'] = annual_freq.sort_values(['STATE', 'YEAR'])['STATE'].values
rolling_avg['YEAR']  = annual_freq.sort_values(['STATE', 'YEAR'])['YEAR'].values

# Merge back
df = df.merge(rolling_avg[['STATE', 'YEAR', 'ROLLING_AVG_3Y']], on=['STATE', 'YEAR'], how='left')

print(df[['STATE', 'YEAR', 'ANNUAL_EVENT_COUNT', 'ROLLING_AVG_3Y']].head(10))

           STATE  YEAR  ANNUAL_EVENT_COUNT  ROLLING_AVG_3Y
0        FLORIDA  2000              1244.0          1244.0
1        FLORIDA  2000              1244.0          1244.0
2        FLORIDA  2000              1244.0          1244.0
3  WEST VIRGINIA  2000              1123.0          1123.0
4    MISSISSIPPI  2000               934.0           934.0
5    MISSISSIPPI  2000               934.0           934.0
6    MISSISSIPPI  2000               934.0           934.0
7          MAINE  2000               728.0           728.0
8          MAINE  2000               728.0           728.0
9          MAINE  2000               728.0           728.0


In [16]:
print(df[df['STATE'] == 'WEST VIRGINIA']['YEAR'].value_counts().sort_index().head())

YEAR
2000    1123
2001     838
2002     889
2003     866
2004     618
Name: count, dtype: int64


## Feature 03: Seasonality

In [17]:
# In EDA, we've discovered that between May and July there's a spike in events, but the model can't use MONTH_NAME directly.

# Map month names to numbers
month_map = {
    'January': 1, 'February': 2, 'March': 3,
    'April': 4, 'May': 5, 'June': 6,
    'July': 7, 'August': 8, 'September': 9,
    'October': 10, 'November': 11, 'December': 12
}

df['MONTH'] = df['MONTH_NAME'].map(month_map)

# Peak season flag (May to July = 1, rest = 0)
df['IS_PEAK_SEASON'] = df['MONTH'].between(5, 7).astype(int)

# Quarter (1 to 4)
df['QUARTER'] = df['MONTH'].apply(lambda x: (x - 1) // 3 + 1)

print(df[['MONTH_NAME', 'MONTH', 'QUARTER', 'IS_PEAK_SEASON']].head(10))

  MONTH_NAME  MONTH  QUARTER  IS_PEAK_SEASON
0   December     12        4               0
1   December     12        4               0
2   December     12        4               0
3   December     12        4               0
4     August      8        3               0
5     August      8        3               0
6     August      8        3               0
7    January      1        1               0
8    January      1        1               0
9    January      1        1               0


## Feature 04: Severity Index

In [ ]:
# In this project, each event has 2 impact metrics: total damage and total deaths. 
# But they are in completely different scales. We can create a composite impact score to capture overall severity.
# The solution: Normalize on a 0 to 1 scale and then combine them.


from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler() # This will scale each feature to the [0, 1] range based on min and max values in the dataset.

# Apply log transformation before normalizing
# log1p = log(x + 1), the +1 avoids log(0) error
df['DAMAGE_LOG'] = np.log1p(df['TOTAL_DAMAGE'])
df['DEATHS_LOG']  = np.log1p(df['TOTAL_DEATHS'])

# Normalize the log-transformed values
df['DAMAGE_NORMALIZED'] = scaler.fit_transform(df[['DAMAGE_LOG']])
df['DEATHS_NORMALIZED']  = scaler.fit_transform(df[['DEATHS_LOG']])

# Severity index
df['SEVERITY_INDEX'] = (
    0.6 * df['DAMAGE_NORMALIZED'] +
    0.4 * df['DEATHS_NORMALIZED']
)

print(df[['TOTAL_DAMAGE', 'TOTAL_DEATHS', 'DAMAGE_NORMALIZED', 'DEATHS_NORMALIZED', 'SEVERITY_INDEX']].head(10))
print(f"\nSeverity index range: {df['SEVERITY_INDEX'].min():.4f} to {df['SEVERITY_INDEX'].max():.4f}")

   TOTAL_DAMAGE  TOTAL_DEATHS  DAMAGE_NORMALIZED  DEATHS_NORMALIZED  \
0           0.0             0           0.000000                0.0   
1           0.0             0           0.000000                0.0   
2           0.0             0           0.000000                0.0   
3           0.0             0           0.000000                0.0   
4        2000.0             0           0.321983                0.0   
5        2000.0             0           0.321983                0.0   
6        1000.0             0           0.292644                0.0   
7           0.0             0           0.000000                0.0   
8           0.0             0           0.000000                0.0   
9           0.0             0           0.000000                0.0   

   SEVERITY_INDEX  
0        0.000000  
1        0.000000  
2        0.000000  
3        0.000000  
4        0.193190  
5        0.193190  
6        0.175586  
7        0.000000  
8        0.000000  
9        0.000000 

In [21]:
# Final feature set
features = [
    'EVENT_ID',
    'STATE',
    'YEAR',
    'MONTH',
    'QUARTER',
    'EVENT_TYPE',
    'ANNUAL_EVENT_COUNT',
    'ROLLING_AVG_3Y',
    'IS_PEAK_SEASON',
    'TOTAL_DAMAGE',
    'TOTAL_DEATHS',
    'SEVERITY_INDEX'
]

df_features = df[features].copy()

# Save to processed folder
df_features.to_csv('../data/processed/features.csv', index=False)

print(f"Dataset saved.")
print(f"Shape: {df_features.shape}")
print(f"\nFeatures created:")
for col in features:
    print(f"  - {col}")

Dataset saved.
Shape: (1520485, 12)

Features created:
  - EVENT_ID
  - STATE
  - YEAR
  - MONTH
  - QUARTER
  - EVENT_TYPE
  - ANNUAL_EVENT_COUNT
  - ROLLING_AVG_3Y
  - IS_PEAK_SEASON
  - TOTAL_DAMAGE
  - TOTAL_DEATHS
  - SEVERITY_INDEX
